# Activation maps

**What it does.** Show which pixels drove a trained classifier's decision.

**When to use it.** After training a classifier that works, to check it is looking at the biology and not at a plate edge or a focus artefact.

**What you get.** Per-class activation overlays and the aggregated attention maps.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.deep_spacr.generate_activation_map`

```
generate_activation_map(settings)
```

Generate saliency or Grad-CAM activation maps for every image in a tar dataset.

In [ ]:
from spacr.deep_spacr import generate_activation_map

## 3. Settings

`spacr.settings.get_default_generate_activation_map_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import get_default_generate_activation_map_settings

defaults = get_default_generate_activation_map_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (str) - What a pixel is replaced with when the deletion/insertion
    # curves remove it: 'blur', 'zero' or 'noise'. It is a confound, not
    # a detail - blanking to zero creates a hard edge the model has
    # never seen, so part of the score drop measures the artefact rather
    # than the lost information. 'blur' is the least out-of-distribution
    # and is the default; comparing two baselines is a fair way to ask
    # how much of your AUC is real. Default 'blur'.
    'attribution_baseline': 'blur',

    # (int) - Points along the deletion and insertion curves used to
    # score a map. At each step the highest-ranked remaining pixels are
    # removed (or added) and the model re-run, so this is the resolution
    # of the area-under-curve that judges whether the map describes what
    # the model actually uses. More steps give a smoother AUC at
    # linearly more forward passes. Default 12.
    'attribution_steps': 12,

    # (int) - How many images are held and processed together in one
    # pass: field stacks during normalization and Cellpose segmentation,
    # crops per step during classifier training and activation maps.
    # Raising it speeds runs up but increases RAM/VRAM roughly linearly;
    # lower it on out-of-memory errors. Defaults: 50 for mask
    # generation, 64 for training.
    'batch_size': 64,

    # (str) - Which attribution map is computed. 'gradcam' weights the
    # target_layer feature maps by their pooled gradients into a coarse
    # heatmap of the region that drove the call; 'gradcam_pp' currently
    # computes the identical map and only changes the output folder and
    # table name. 'saliency_image' sums the absolute input gradient into
    # one map; 'saliency_channel' keeps it per channel so you can see
    # which stain mattered. Default 'gradcam'.
    'cam_type': 'gradcam',

    # (list of int) - Zero-indexed image channels kept in merged/*.npy
    # and measured by measure_crop; each entry produces its own
    # <object>_channel_<n>_* intensity columns. The list length fixes
    # where masks land, so cell/nucleus/pathogen_mask_dim must shift if
    # you change it. Preprocessing silently resets it to range(n) when
    # it does not match the number of channel folders found. Default
    # [0,1,2,3].
    'channels': [1, 2, 3],

    # (bool) - Correlate every input channel against every
    # activation-map channel per image and write the result to the
    # <cam_type>_correlations table: a Pearson coefficient plus Manders
    # M1/M2 at each manders_thresholds percentile (15, 50, 75 by
    # default). Use it to quantify which stain the model attends to
    # instead of eyeballing heatmaps; it needs save=True to reach the
    # database. Default True.
    'correlation': True,

    # (str) - Path to the .tar archive of single-object PNG crops
    # produced by generate_dataset, which the activation-map step opens
    # with TarImageDataset. The plate folder is inferred two levels
    # above it and CAM outputs are written next to it under
    # <tar_name>/<cam_type>/. Must be a full path, not just a file name.
    # Default ''.
    'dataset': 'path',

    # (str) - The 'absence of signal' image integrated gradients
    # integrates away from: 'zero' is black, 'blur' is your own image
    # blurred, 'noise' is random. This choice IS the explanation's
    # reference point and changes the result - a black baseline
    # attributes to everything bright, which on dark-field microscopy
    # means it attributes to the object merely for existing. 'blur'
    # keeps the low-frequency content and asks what the detail
    # contributes. Default 'zero'.
    'ig_baseline': 'zero',

    # (int) - Interpolation steps between the baseline and your image
    # for integrated gradients. The method's guarantee - that the
    # attributions sum to the score difference - only holds in the
    # limit, so too few steps silently breaks it; 50 is the usual
    # default and the completeness error is worth checking if you lower
    # it. Cost is linear in this number. Default 50.
    'ig_steps': 50,

    # (int) - Side length in pixels of the centre crop taken from each
    # object PNG before it reaches the model. Images are cropped, not
    # rescaled, so a larger value zero-pads and a smaller one throws
    # away the object's edges. It is also the resolution the backbone is
    # built at, which matters for ViT/Swin/inception. Match it to the
    # crop size used when the dataset was generated. Default 224.
    'image_size': 224,

    # (list) - Percentiles (0-100) at which Manders' overlap
    # coefficients are computed. For each object, each entry thresholds
    # both channels at that percentile; pixels above both count as
    # overlap, and M1/M2 report each channel's fraction of total object
    # intensity there, saved as M1_correlation_<t> and
    # M2_correlation_<t>. High values isolate the brightest puncta.
    # Requires calculate_correlation. Default [15, 85, 95].
    'manders_thresholds': [15, 50, 75],

    # (str) - Path to a trained spaCR classifier saved as a whole
    # PyTorch object (loaded with torch.load(weights_only=False), not a
    # state_dict). Used when applying a model to a dataset tar and when
    # generating activation maps. deep_spacr overwrites it with the
    # freshly trained model whenever train is True, so set it only to
    # score with an existing model. Default ''.
    'model_path': 'path',

    # (str) - Backbone architecture for the single-object image
    # classifier, passed to choose_model: any TorchVision classification
    # model name (resnet50, maxvit_t, densenet121, ...). An unrecognised
    # name is not fatal at call time - choose_model prints 'Invalid
    # model_type' and returns None, so training then fails; the special
    # name 'custom' passes the name check but raises
    # NotImplementedError. Bigger backbones capture subtler phenotypes
    # but cost VRAM and epochs, and the name becomes part of the output
    # model folder path (src/model/<model_type>/...). Default 'maxvit_t'
    # in the training pipelines; the activation-map tool defaults to
    # 'maxvit', and only that exact string triggers its automatic
    # target-layer pick; the Tk/Qt combo preselects 'resnet50'.
    'model_type': 'maxvit',

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where
    # -1 means every core. Raise it to shorten CPU-bound steps until RAM
    # or disk I/O saturates. Note the measure-and-crop pipeline
    # overrides your value with cpu_count()-4. Defaults vary by
    # pipeline: cpu_count()-4, -1, or None.
    'n_jobs': None,

    # (bool) - Percentile-normalize each image channel (2nd to 98th
    # percentile, clipped to 0-1) before display or model input; in the
    # activation-map tool this rescales the image the CAM/saliency
    # heatmap is drawn over. Turn it on when raw channels are too dim to
    # read under the overlay. Affects display and input scaling only,
    # never stored pixels. Default True.
    'normalize': True,

    # (bool) - Apply the same per-channel mean=0.5, std=0.5
    # normalisation used during training to each image before it enters
    # the model when generating activation maps. Keep it matched to how
    # the model was trained, otherwise inputs are off-distribution and
    # both the predicted classes and the maps are meaningless. Distinct
    # from 'normalize', which only percentile-stretches images for
    # display. Default True.
    'normalize_input': True,

    # (str) - Which mask decides where an object is when the pointing
    # game scores an attribution map: 'cell', 'nucleus', 'pathogen' or
    # 'cytoplasm'. The pointing game asks only whether the map's single
    # hottest pixel lands inside that mask, so it is cheap and coarse -
    # it says nothing about the rest of the map, and a method can score
    # 1.0 while attributing nonsense everywhere else. Default 'cell'.
    'object_type': 'cell',

    # (int) - How far the occlusion patch moves between evaluations.
    # Equal to occlusion_window it tiles without overlap and is fastest;
    # half of it doubles the passes and halves the blockiness. A stride
    # larger than the window leaves unmeasured gaps that appear as an
    # artificial grid in the map. Default 4.
    'occlusion_stride': 4,

    # (int) - Side length in pixels of the patch occlusion slides over
    # the image, blanking it and recording how far the model's score
    # falls. Larger windows are faster and blurrier and will miss a
    # feature smaller than the window; smaller ones resolve fine
    # structure at quadratically more forward passes. Occlusion is the
    # only method here that needs no gradients at all, which is why it
    # is worth its cost as a cross-check on the gradient family. Default
    # 8.
    'occlusion_window': 8,

    # (bool) - In the batch-grid figures, draw the activation map in the
    # 'jet' colormap at 50 percent alpha over the source image. Turn it
    # off and the grid tiles are left empty apart from the
    # predicted-class label, so keep it on whenever plot is enabled. It
    # never affects the per-object activation PNGs saved to disk, which
    # are always the bare map. Default True.
    'overlay': True,

    # (bool) - Render and save QC figures while the pipeline runs:
    # channel montages and Cellpose mask overlays during segmentation,
    # before/after filtration views and crop grids during measurement.
    # It adds figures per batch, so a full plate becomes much slower and
    # more memory-hungry; keep it for small or test_mode runs, which
    # force it on. Default False.
    'plot': False,

    # (bool) - Randomise the model's weights layer by layer and
    # re-attribute, then report how similar the map stays. A method that
    # produces nearly the same picture for a randomised model is an edge
    # detector, not an explanation - and measured on a small CNN the
    # whole CAM family, including the Grad-CAM spaCR defaults to, fails
    # this while saliency and integrated gradients pass. The number is
    # reported for YOUR model rather than assumed, which is the point.
    # Costs one extra attribution per randomised layer. Default True.
    'sanity_check': True,

    # (bool or list of bool) - Whether to save masks to disk. Can be a
    # list of three booleans for [cell, nucleus, pathogen]
    # independently.
    'save': True,

    # (bool) - Shuffle the tar dataset in the DataLoader when generating
    # activation maps, so each batch-grid PDF shows a mixed sample
    # rather than consecutive files from one plate or class. Set False
    # for a deterministic, file-order pass you can line up against the
    # dataset listing. Default True.
    'shuffle': True,

    # (int) - Noisy copies of the image averaged into one attribution
    # map. A single map is dominated by the gradient's local jitter, so
    # 8-50 samples smooth it into something stable enough to compare
    # between images; 0 (the default) runs the method once and is what
    # you want while you are still choosing a method, since it costs one
    # forward-backward pass instead of N. Applies to every method,
    # including the CAM family, where it is averaged explicitly rather
    # than through captum. Default 0.
    'smoothgrad_samples': 0,

    # (float) - Standard deviation of the noise SmoothGrad adds, as a
    # fraction of the image's intensity range. Too small and every
    # sample is the same map, so averaging changes nothing; too large
    # and the samples are of images the model has never seen, so the
    # average describes the model's behaviour on noise rather than on
    # your data. 0.1-0.2 is the usual band. Ignored when
    # smoothgrad_samples is 0. Default 0.15.
    'smoothgrad_sigma': 0.15,

    # (str) - Dotted attribute path to the convolutional layer whose
    # activations and gradients Grad-CAM hooks, e.g.
    # 'base_model.blocks.3.layers.1.layers.MBconv.layers.conv_b';
    # utils.recommend_target_layers(model) lists valid names. Later
    # layers give class-specific but coarse maps, earlier ones finer
    # detail. Required for 'gradcam'/'gradcam_pp' - it is auto-filled
    # only when model_type is exactly 'maxvit', and left None it raises.
    # Default None.
    'target_layer': None,

}

# Fill in anything left unset, then check the source path.
settings = get_default_generate_activation_map_settings(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
generate_activation_map(settings)

## Where the output went

Per-class activation overlays and the aggregated attention maps.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.